In [ ]:
# install libraries

!pip -q install transformers sentence-transformers

# import lib
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel, pipeline
import re
from tqdm import tqdm  # for progress bars
import html
from sentence_transformers import SentenceTransformer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device) # use T4 GPU for faster testing

Using device: cuda


In [ ]:
# connect to google folder and unzip file
from google.colab import drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/BT4222_Group11/scraped_anime.csv.zip"

import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

!ls /content/
# find the csv file inside


Mounted at /content/drive
drive  __MACOSX  sample_data  scraped_anime.csv


In [ ]:
import pandas as pd

csv_path = "/content/scraped_anime.csv"  # the path after unzipping

# Load only the columns for review analysis
df = pd.read_csv(csv_path,
                 usecols=['anime_id', 'review_text', 'helpful_count'])

print("Loaded columns:", df.columns.tolist())
print("Number of reviews:", len(df))
print("Number of unique anime:", df['anime_id'].nunique())
df.head()

Loaded columns: ['anime_id', 'helpful_count', 'review_text']
Number of reviews: 175260
Number of unique anime: 9108


,anime_id,helpful_count,review_text
0,1,1837,People who know me know that I'm not a fan of ...
1,1,1314,Cowboy Bebop is one of those series that is ju...
2,1,432,Cowboy Bebop was originally an anime series cr...
3,1,382,Only a few anime series or movies could be con...
4,1,295,Cowboy Bebop was one of the first anime that t...


In [ ]:
# Cleaning & Sampling
import html
import re

# Cleaning
# Step 1: Initial cleaning
df = df.dropna(subset=['review_text'])

# Step 2: Basic HTML(scraped) cleaning
df['review_clean'] = df['review_text'].str.replace(r'<br\s*/?>', ' ', regex=True)\
                                     .str.replace(r'\n+', ' ', regex=True)\
                                     .str.strip()

# Step 3: Define text cleaning functions. (but this one doesn't have a common pattern to do cleaning?? can only hardcode)
def decode_html(text):
    """Decode HTML entities like &amp; to &"""
    return html.unescape(text)

def remove_metadata_footer(text):
    """Remove ratings/scoring metadata from review text"""
    if not isinstance(text, str):
        return ""

    # Remove everything after separator line (10+ dashes)
    text = re.split(r'-{10,}', text)[0]

    # Remove specific footer patterns
    text = re.sub(r'This review is the final result of.*?club\.', '', text, flags=re.DOTALL)
    text = re.sub(r'Here are their individual scorings.*?Overall - [\d\., ]+', '', text, flags=re.DOTALL)
    text = re.sub(r'Helpful read more$', '', text)

    return text.strip()

# Step 4: Advanced cleaning using functions defined
df['review_clean'] = df['review_clean'].apply(decode_html)\
                                       .apply(remove_metadata_footer)

# Step 5: Remove empty reviews after cleaning
df = df[df['review_clean'].str.len() > 20]

# Sampling (5 reviews per anime)
sampled_df = (
    df.sample(frac=1, random_state=42)
      .groupby('anime_id', group_keys=False)
      .head(5)
      .reset_index(drop=True)
)

print(f"Original reviews: {len(df)}")
print(f"Sampled reviews: {len(sampled_df)}")
print(f"Unique anime covered: {sampled_df['anime_id'].nunique()}")

Original reviews: 175105
Sampled reviews: 29576
Unique anime covered: 9108


In [ ]:
# Sentiment Analysis
# !! need colab (Free T4 GPU)
from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
)

def get_sentiment_score(res):
    return res["score"] if res["label"] == "POSITIVE" else 1 - res["score"]

texts = sampled_df["review_clean"].tolist()

all_out = []
batch_size = 64
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    out = sentiment_pipe(batch, truncation=True, max_length=512)
    all_out.extend(out)

sampled_df["sentiment"] = [get_sentiment_score(r) for r in all_out]

print(f"Done! Sentiment range: {sampled_df['sentiment'].min():.2f} to {sampled_df['sentiment'].max():.2f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Done! Sentiment range: 0.00 to 1.00


In [ ]:
# BERT Embedding
# !! need colab (Free T4 GPU)
from sentence_transformers import SentenceTransformer

st_device = "cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("bert-base-nli-mean-tokens", device=st_device)
model = SentenceTransformer("all-MiniLM-L6-v2", device=st_device)  # changed to this one, much faster

batch_size=128  # T4 can handle this easily for MiniLM

embeddings = model.encode(
    sampled_df['review_clean'].tolist(),
    batch_size=batch_size,
    show_progress_bar=True
)

sampled_df['embedding'] = list(embeddings)

print(f"Done! Embedding dimension: {embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/232 [00:00<?, ?it/s]

Done! Embedding dimension: 384


In [ ]:
# Aggregate to anime level for review features

# Sentiment stats
review_sentiment = (
    sampled_df.groupby("anime_id", as_index=False)
    .agg(
        sentiment_mean=("sentiment", "mean"),
        sentiment_std=("sentiment", "std"),
        review_count=("sentiment", "count"),
    )
)
review_sentiment["sentiment_std"] = review_sentiment["sentiment_std"].fillna(0)


# Helpfulness stats
review_metadata = sampled_df.groupby('anime_id').agg(
    avg_helpfulness=('helpful_count', 'mean'),
    total_helpfulness=('helpful_count', 'sum'),
    helpfulness_std=('helpful_count', 'std')
).reset_index()
review_metadata['helpfulness_std'] = review_metadata['helpfulness_std'].fillna(0)

# Weighted sentiment (using helpful_count as weights)
def weighted_mean(group):
    weights = group['helpful_count']
    values = group['sentiment']

    # Handle case where all helpful_count are 0 (avoid div by zero)
    if weights.sum() == 0:
        return values.mean()  # Fall back to unweighted
    return np.average(values, weights=weights)

weighted_sentiment = (
    sampled_df.groupby("anime_id")
    .apply(weighted_mean, include_groups=False)
    .reset_index(name="sentiment_mean_weighted")
)

# Merge with your existing sentiment stats
review_sentiment = review_sentiment.merge(
    weighted_sentiment, on="anime_id", how="left"
)

# Average embeddings per anime
review_embeddings = sampled_df.groupby('anime_id')['embedding'].apply(
    lambda x: np.mean(np.vstack(x), axis=0)
)

print("Done!")
print(f"Anime with review embeddings: {len(review_embeddings)}")

Done!
Anime with review embeddings: 9108


In [ ]:
# Save output

out_dir = "/content/drive/MyDrive/BT4222_Group11/review_features"
import os
os.makedirs(out_dir, exist_ok=True)

# Create embedding matrix from the grouped data
embedding_matrix = np.vstack(review_embeddings.values)
anime_ids = review_embeddings.index.values

print(f"Embeddings shape: {embedding_matrix.shape}")  # Should be (9108, 384)
print(f"Anime IDs count: {len(anime_ids)}")           # Should be 9108

np.save(f"{out_dir}/review_bert_embeddings.npy", embedding_matrix)
np.save(f"{out_dir}/review_anime_ids.npy", anime_ids)

review_sentiment.to_csv(f"{out_dir}/review_sentiment_stats.csv", index=False)
review_metadata.to_csv(f"{out_dir}/review_derived_metadata.csv", index=False)

Embeddings shape: (9108, 384)
Anime IDs count: 9108
